# 第 06 章　函数

> **本章定位**：把重复的逻辑封成"有名有姓、可传参、可返回"的单元。这是从"写脚本"到"写程序"的分界。
>
> **知识点范围**：`def` 定义与调用、位置参数、关键字参数、默认参数、可变参数 `*args`、`return` 与多返回值、局部变量与全局变量、函数嵌套调用、递归（终止条件 + 递推关系）。
>
> **本章约定**：可用前 5 章全部内容（含列表、字典）；**不引入** `lambda`、`map`、`filter`、装饰器（第 7 章），不引入 `import`（第 10 章）。

> **先修知识**：第 01—05 章（变量、判断与循环、列表与字典）。函数体内部会大量使用这些容器与循环，本章不再复述其语法。
>
> **学习目标**：完成本章后，你能够：
>
> 1. 定义 1 个含 2 个位置参数的函数并调用 3 次，打印每次的实参与返回值；
> 2. 写出同时使用位置、关键字、默认与可变参数的函数，说明 4 类参数与形参的对应关系；
> 3. 用 `return` 返回 2 个值并在调用处解包，分别打印两个结果；
> 4. 先预测再运行 6 个变量在函数内外的可见性，判定每个变量是局部还是全局；
> 5. 写出含终止条件与递推关系的递归函数，打印 5 层调用的中间值并核对层数。

---

## 本章学习路径

| 阶段 | 内容 | 编号 | 作用 |
|---|---|---|---|
| 第一部分 | 知识讲解 | 知识点 6.1 — 6.5 | 定义、参数、返回、作用域、递归 + 随堂示例 |
| 第二部分 | 代码示范 | 案例 6-1、案例 6-2 | 函数库 / 递归统计 |
| 第三部分 | 布置练习 | 练习 6-1、练习 6-2 | 可变参数计费 / 重构播报 |
| 第四部分 | 习题答案 | 答案 6-1、答案 6-2 | 对照解析与评分要点 |

**知识点与代码对照表**：

| 知识点 | 名称 | 讲解位置 | 代码示范 | 对应练习 |
|---|---|---|---|---|
| 6.1 | `def` 定义、调用与文档字符串 | 6.2 | 案例 6-1、案例 6-2 | 练习 6-1、练习 6-2 |
| 6.2 | 四类参数（位置 / 关键字 / 默认 / `*args`） | 6.3 | 案例 6-1 | 练习 6-1 |
| 6.3 | `return`、多返回值与提前返回 | 6.4 | 案例 6-1、案例 6-2 | 练习 6-2 |
| 6.4 | 局部变量与全局变量 | 6.5 | 案例 6-1（对比） | —— |
| 6.5 | 递归与终止条件 | 6.6 | 案例 6-2 | —— |

## 教学提示（教师用）

- **建议课时**：8 学时（4 次课），对应课程标准模块 6。
- **讲解主线**：函数只做三件事——收参数、做事情、给结果。先讲"为什么要拆"（重复与一致性），再讲四类参数，最后讲递归；递归务必先画调用图。
- **学生常见卡点**：忘记 `return` 导致调用结果为 `None`；默认参数写成可变对象（如 `[]`）被多次调用共享；局部变量与全局变量同名引起误判；递归缺终止条件触发 `RecursionError`。
- **课堂活动建议**：把第 3 章的拥堵分级原样改写成函数版，两版并排对比，让学生指出"改一处要动几行"的差别。

---

# 第一部分　知识讲解

## 6.1 场景导入：重复与一致性

同一个"拥堵判定"逻辑，如果在 5 个地方各写一遍，改动一次就要改 5 处；同一个"通行能力"公式，如果每次手写 `2200 * lanes`，把常量写错也无人发现。

函数解决的正是**重复**与**一致性**问题。判断"该不该抽函数"的标准很简单：**这段逻辑会不会被用第二次**，以及**能否用一句话说清它做什么**。

## 6.2 知识点 6.1　定义、调用与文档字符串

### 6.2.1 核心概念

- `def 函数名(参数表):` 开始定义，函数体缩进；
- 定义时**不执行**，只在内存中登记；**调用**（写 `函数名(实参)`）时才执行；
- 函数名后用圆括号，即使无参数也要写 `()`；
- 函数第一行可用三引号写**文档字符串**，说明输入、输出与职责。

**原理补充（概念深化）**：解释器执行到 `def` 时并不运行函数体，而是把函数体编译成代码对象，再把函数名绑定到函数对象并登记进名字空间；函数体里哪怕写着 `print`，此刻也一行都不执行。只有遇到 `函数名(实参)` 才新建栈帧，把实参绑到形参逐条执行，返回后栈帧即销毁。这解释了「先定义后调用」不是习惯而是自上而下绑定名字的必然：调用时名字必须已存在，否则抛 `NameError`。文档字符串本质是 `__doc__` 属性——编译器只在函数体首条语句是字符串字面量时才收进它，写在别处就被丢弃。常见误解是把形参当成「变量声明」：它不分配存储，只是调用时绑定的名字；函数名 `capacity` 与普通变量同处一张名字表，名字仅是指向函数对象的一个引用。

### 6.2.2 语法要点

| 要素 | 规范 | 示例 |
|---|---|---|
| 定义 | `def 名(参数):` | `def capacity(lanes, per_lane=2200):` |
| 文档字符串 | 紧跟 `def` 行，用三引号 | `"""理论通行能力（辆/h）"""` |
| 函数体 | 缩进 4 空格 | 内部可写任意语句 |
| 调用 | `函数名(实参)` | `capacity(3)` |
| 查看文档 | `函数名.__doc__` | 第 10 章会用于模块说明 |

### 6.2.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记调用 | 定义了却没有任何输出 | 定义后必须写调用语句 |
| 调用写在定义之前 | `NameError: name 'xxx' is not defined` | 先定义后调用（脚本自上而下执行） |
| 缩进不统一 | `IndentationError` | 函数体统一 4 空格 |
| 函数名后用中括号 | `capacity[3]` 报 `TypeError` | 调用用圆括号 |
| 命名不规范 | `def f(x):` | 动词或名词短语，小写下划线 |

### 6.2.4 随堂示例：从定义到调用的最小函数

**核心要点**：验证 `def` 定义、按位置传参、默认参数生效，以及文档字符串可通过 `函数名.__doc__` 读回。

In [ ]:
def capacity(lanes, per_lane=2200):
    """计算理论通行能力：车道数 × 单车道能力"""
    return lanes * per_lane

print("3 车道：", capacity(3))                 # 使用默认 per_lane
print("2 车道（匝道 1800）：", capacity(2, 1800))  # 覆盖默认值
print("函数说明：", capacity.__doc__)

**预期输出**：

```text
3 车道： 6600
2 车道（匝道 1800）： 3600
函数说明： 计算理论通行能力：车道数 × 单车道能力
```

**变式代码（迁移练习）**：相对随堂示例直接返回并打印 `__doc__`，本变式在函数体内留一条执行痕迹 `print`，用来验证「`def` 只登记不执行、调用才执行」，并观察两次调用各自独立触发一次函数体。

In [ ]:
# 定义阶段：函数体含 print，但此刻不会输出任何内容
def signal_cycle(lanes, base=30):
    print("  [函数体执行] 车道", lanes)
    return base + lanes * 5
print("准备调用")
print("结果 A：", signal_cycle(2))
print("结果 B：", signal_cycle(3, 20))

**预期输出**：

```text
准备调用
  [函数体执行] 车道 2
结果 A： 40
  [函数体执行] 车道 3
结果 B： 35
```

### 6.2.5 实操步骤（上机跟随）

1. 先只保留 `def` 段单独运行，屏幕没有任何输出，说明定义阶段只登记名字、函数体一行未执行
2. 补上调用语句后先打印 `准备调用`，随后才出现 `[函数体执行] 车道 2`，证明调用才触发函数体
3. 核对 `结果 A： 40`（base 默认 30 加 2×5）与 `结果 B： 35`（覆盖 base 为 20 后 20 加 3×5）
4. 两次调用各打印一次函数体行，说明每次调用新建独立栈帧、彼此不共享状态
5. 把调用语句移到 `def` 之前运行，预期报 `NameError`，反证脚本自上而下绑定名字

### 6.2.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 定义了函数却没有任何输出 | 只写了 `def` 没写调用语句，函数体不会自行执行 | 在定义之后补一行 `函数名(实参)` |
| 调用写在定义之前报 `NameError` | 脚本自上而下执行，调用时名字尚未绑定到函数对象 | 把 `def` 移到调用语句之前 |
| 用 `capacity[3]` 取值报 `TypeError` | 把圆括号调用误写成中括号下标，函数对象不支持下标 | 调用一律用圆括号 `capacity(3)` |
| `__doc__` 打印出 `None` | 文档字符串没有紧贴 `def` 行写在函数体第一条 | 把说明字符串放到函数体最开头（概念见原理补充） |
| 函数体缩进忽多忽少报 `IndentationError` | 同一函数体内混用了不同空格数 | 统一用 4 个空格缩进 |

## 6.3 知识点 6.2　四类参数

### 6.3.1 核心概念

| 参数类型 | 写法 | 调用方式 | 用途 |
|---|---|---|---|
| 位置参数 | `def density(cars, length_km, lanes)` | `density(96, 2.5, 3)` | 按顺序传，最常用 |
| 关键字参数 | 同上 | `density(cars=96, lanes=3, length_km=2.5)` | 顺序无关，可读性强 |
| 默认参数 | `def capacity(lanes, per_lane=2200)` | `capacity(3)` 或 `capacity(3, 1800)` | 常用值设默认，简化调用 |
| 可变参数 | `def total_flow(*flows)` | `total_flow(820, 1100, 760)` | 参数个数不定 |

**原理补充（概念深化）**：编译函数签名时，解释器按从左到右为每个形参分配绑定槽：调用时先用位置实参依次填满，剩余没填到的只能靠默认值兜底。一旦某个形参带默认值，它右边的形参就必须同样「没有实参也有值」，否则 `f(1)` 给谁会产生歧义，CPython 干脆在语法层禁止，报 `non-default argument follows default argument`。从内存视角看，默认值只在定义时求值一次，存进函数对象的 `__defaults__` 元组，每次调用都复用同一份——这正是「默认值用可变列表会跨调用串味」的根源。`*args` 的星号是「打包」：把多余位置实参收成元组，能 `sum()` 却不能 `append()`；调用处写 `*序列` 则是「拆包」。常见误解：关键字参数并非另一种类型，只是「按名字绑定」到同名形参；`*args` 之后的参数必须显式写名字，因为已无法按位置捕获。

### 6.3.2 语法要点（必须记住的三条规则）

1. **默认参数必须放在参数表末尾**：`def f(a, b=1)` 合法，`def f(a=1, b)` 是 `SyntaxError`。
2. **`*args` 在函数内部是元组**，可 `len()`、`sum()`、遍历、下标访问。
3. **位置参数与关键字参数可混用，但位置参数必须在前面**：`fee(120, discount=0.95)` 合法；`fee(km=120, 0.95)` 是语法错误。

### 6.3.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 默认参数写在前面 | `SyntaxError: non-default argument follows default argument` | 移到末尾 |
| 忘记传必需参数 | `TypeError: missing 1 required positional argument` | 补参数或设默认值 |
| 关键字拼错 | `TypeError: got an unexpected keyword argument` | 对照定义检查参数名 |
| `*args` 当列表用 | `flows.append(...)` 报错 | 元组不可变，需要列表先 `list(flows)` |
| 可变默认参数用列表 | 多次调用共享同一个列表（进阶坑） | 默认值用 `None`，函数内再建列表 |

### 6.3.4 随堂示例：四类参数放在同一个函数里

**核心要点**：验证位置参数、默认参数、可变参数 `*extra`、关键字参数 `discount=` 一次协作：`*extra` 把多余的位置实参收成元组，关键字实参必须写在最后。

In [ ]:
def fee(km, rate=0.5, *extra, discount=1.0):
    """位置 + 默认 + 可变 + 关键字，四类参数一次演示"""
    print("里程：", km, "| 费率：", rate)
    print("额外标记：", extra, "| 折扣：", discount)
    return round(km * rate * discount, 2)

print("默认调用：", fee(120))
print("完整调用：", fee(120, 0.6, "夜间", "ETC", discount=0.95))

**预期输出**：

```text
里程： 120 | 费率： 0.5
额外标记： () | 折扣： 1.0
默认调用： 60.0
里程： 120 | 费率： 0.6
额外标记： ('夜间', 'ETC') | 折扣： 0.95
完整调用： 68.4
```

**变式代码（迁移练习）**：相对随堂示例把四类参数集中打印，本变式迁移为桥梁计费：验证 `*extras` 作为元组可直接 `sum`、关键字调用可打乱顺序，以及默认值在不传附加费时生效。

In [ ]:
# 通行费：km 与 rate 位置/默认，*extras 收多余附加费，discount 关键字参数
def bridge_fee(km, rate=0.5, *extras, discount=1.0):
    surcharge = sum(extras)          # *extras 是元组，可直接 sum
    return round((km * rate + surcharge) * discount, 2)
print("默认（无附加）：", bridge_fee(120))
print("位置+附加费：", bridge_fee(120, 0.6, 10, 5))
print("关键字调用（只改折扣）：", bridge_fee(km=120, discount=0.9, rate=0.5))

**预期输出**：

```text
默认（无附加）： 60.0
位置+附加费： 87.0
关键字调用（只改折扣）： 54.0
```

### 6.3.5 实操步骤（上机跟随）

1. 不传附加费时 `extras` 为空元组，`sum(extras)` 为 0，故 `bridge_fee(120)` 输出 60.0
2. 追加两个位置实参 10、5 被 `*extras` 打包成 `(10, 5)`，`sum` 得 15，输出 87.0
3. 把 `km/rate/discount` 全部写成关键字、顺序与定义不同，仍得 54.0，验证按名绑定
4. 在关键字实参后再补位置实参，预期 `SyntaxError`，验证位置参数必须在关键字之前
5. 把 `*extras` 改当列表 `append` 会报 `AttributeError`，记住它本质是元组、要列表先 `list()`

### 6.3.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `SyntaxError: non-default argument follows default argument` | 带默认值的形参后面又跟了无默认值形参 | 把所有带默认值的形参移到参数表末尾 |
| `AttributeError: 'tuple' object has no attribute 'append'` | 把 `*args` 当成列表使用 | 记住 `*args` 是元组，要列表先 `list(args)` |
| 多传位置参数报 `TypeError` | 既无对应形参、也没有 `*args` 接收 | 补上形参或加 `*args` 兜底 |
| 关键字拼错报 `got an unexpected keyword argument` | 关键字按名字绑定，须与 `def` 形参名完全一致 | 对照定义逐字检查参数名 |
| 默认值用 `[]` 后多次调用数据串味 | 默认值在定义时求值一次并被所有调用共享 | 默认值改用 `None`，函数内再新建列表 |

## 6.4 知识点 6.3　返回值

### 6.4.1 核心概念

- `return 值` 把结果交回调用处，同时**立即结束函数**；
- 无 `return` 或只写 `return` 时，函数返回 `None`；
- `return a, b, c` 实际返回**元组**，可用 `x, y, z = f()` 解包。

**"提前返回"技巧**：多分支判定时，每个分支直接 `return`，最后一个分支不必写 `else`，代码更平。

**原理补充（概念深化）**：函数执行到 `return` 时，先把表达式求值结果放进当前栈帧的返回值槽，随后立即销毁该栈帧、把控制权交回调用处；因此同一函数里 `return` 之后的语句永远不会运行，这就是「提前返回」能省掉 `else` 的底层原因。`return a, b, c` 里的逗号才是元组构造器，圆括号可省，解释器实际只返回一个对象——元组；`x, y, z = f()` 是序列解包，按位置一一绑定，变量数不符就抛 `ValueError`。`print` 与 `return` 的分工常被混淆：前者只把文本写进标准输出流，不产生可继续计算的值；函数若无 `return`，解释器会在函数体末尾隐式 `return None`，于是 `print(f())` 显示 `None`。常见误解是以为返回的是「屏幕上的字」或几个独立变量，其实返回的是对象的引用，调用方拿到后能继续参与运算、存进列表或再次传参。

### 6.4.2 语法要点

| 写法 | 返回 | 说明 |
|---|---|---|
| `return lanes * per_lane` | 数值 | 最常见 |
| `return "畅通"` | 字符串 | 分支返回不同等级 |
| `return a, b, c` | 元组 `(a, b, c)` | 多返回值 |
| `s, d, lv = report(...)` | 解包 | 变量数须与返回值数一致 |
| 无 `return` | `None` | `print(f(...))` 会显示 `None` |

### 6.4.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 用 `print` 代替 `return` | 调用处拿不到结果，只能看到屏幕输出 | 需要参与计算就必须 `return` |
| `return` 写在循环内导致提前结束 | 只处理了第一条数据 | 想收集全部结果应先累积再 `return` |
| 解包数量不匹配 | `ValueError: too many values to unpack` | 变量数 = 返回值数 |
| 以为返回多个值就是"多个返回值" | 实际是一个元组 | 记住本质是元组，可整体接收 |

### 6.4.4 随堂示例：返回多个值并按位置解包

**核心要点**：验证 `return a, b, c` 实际返回一个元组，调用方可用三个变量一次解包；也可用下标只取其中一个。

In [ ]:
def report(road, cars, length_km, lanes):
    """返回三个值：站点、密度、等级"""
    d = round(cars / (length_km * lanes), 2)
    level = "畅通" if d < 10 else "缓行"
    return road, d, level

site, dens, level = report("京港澳 K12", 96, 2.5, 3)
print("站点：", site)
print("密度：", dens, "辆/(km·车道)")
print("等级：", level)

print("只取密度：", report("京开 K8", 30, 2, 2)[1])

**预期输出**：

```text
站点： 京港澳 K12
密度： 12.8 辆/(km·车道)
等级： 缓行
只取密度： 7.5
```

**变式代码（迁移练习）**：相对随堂示例返回三元组并按位置解包，本变式改用「提前返回」分支，并对比「单个变量整体接住元组」与「三变量解包」两种接收方式。

In [ ]:
# 提前返回：命中分支直接结束，后面语句不再执行
def judge(plate, km, minutes):
    v = km / (minutes / 60)
    if v > 120:
        return plate, round(v, 1), "超速"
    return plate, round(v, 1), "正常"
whole = judge("京B67890", 10, 4)      # 一个变量整体接住，实际是元组
print("整体接收：", whole, "| 下标取首元素：", whole[0])
p, v, lv = judge("京A12345", 10, 6)  # 按位置解包成三个变量
print("解包结果：", p, v, lv)

**预期输出**：

```text
整体接收： ('京B67890', 150.0, '超速') | 下标取首元素： 京B67890
解包结果： 京A12345 100.0 正常
```

### 6.4.5 实操步骤（上机跟随）

1. `judge("京B67890", 10, 4)` 算得 150 km/h，命中 `if` 提前 `return`，其后那条 `return` 不再执行
2. 用单个变量 `whole` 接住得到 `('京B67890', 150.0, '超速')`，`whole[0]` 能取车牌，证明多返回值本质是一个元组
3. `judge("京A12345", 10, 6)` 得 100 km/h 走最后一条 `return`，解包为 `p/v/lv` 三个变量并打印
4. 把解包改成两个变量，预期 `ValueError: too many values to unpack`，验证变量数须等于元素数
5. 删掉 `return` 只留 `print` 后调用处将得到 `None`，说明屏幕输出不等于返回值

### 6.4.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `print(f(...))` 显示 `None` | 函数体没有 `return`，解释器在末尾隐式返回 `None` | 需要结果参与计算就补 `return 值` |
| `ValueError: too many values to unpack` | 解包变量数与元组元素数不一致 | 让变量数等于返回值数，或整体接收后按下标取 |
| 循环里只处理了第一条数据 | `return` 写在 `for` 内，第一次迭代就结束函数 | 先累积到列表，循环结束后再 `return` |
| 以为返回了三个独立值 | 逗号构造的是元组，函数只返回一个对象 | 记住 `return a, b, c` 等价于 `return (a, b, c)` |
| `return` 之后的调试语句从不执行 | `return` 立即销毁栈帧并交回控制权 | 把要执行的语句放到 `return` 之前 |

## 6.5 知识点 6.4　局部变量与全局变量

### 6.5.1 核心概念

- **局部变量**：在函数内部第一次赋值的变量，只在该函数内有效，函数结束即消失；
- **全局变量**：在模块顶层（函数外）定义的变量，函数内**可读不可改**；
- 在函数内给同名变量赋值，会**新建一个局部变量**，不影响全局变量。

**原理补充（概念深化）**：作用域本质是名字查找规则，而不是内存分区。CPython 编译函数体时先扫描全部赋值语句，凡在函数内被 `=` 赋值的名字一律登记为局部变量（写进 `co_varnames`），这个判定在编译期就定死，与运行先后无关。名字查找按 LEGB 逐层进行：Local、Enclosing、Global、Built-in，函数内赋值永远只在最内层 Local 绑定，不会「穿过」去改全局。这也解释了那个反直觉的报错：函数里先读 `total` 再给它赋值，会因同名已判为局部、读取时局部槽还没值而抛 `UnboundLocalError`。常见误解是「函数内赋值会修改全局变量」——其实只是新建同名局部绑定，全局引用纹丝不动。每次调用的栈帧彼此独立，局部变量随栈帧销毁，故跨函数传递状态的正解是参数进、返回值出。

### 6.5.2 语法要点

| 场景 | 结果 |
|---|---|
| 函数内读全局变量 | 正常读到 |
| 函数内给同名变量赋值 | 新建局部变量，全局不变 |
| 函数内 `global x` 后再赋值 | 修改全局变量（**本章不推荐**，破坏可测试性） |
| 函数内引用未赋值的同名变量 | `UnboundLocalError` |

### 6.5.3 常见用法与易错点

- **优先用参数与返回值传递数据**，而不是靠全局变量"隔着函数通信"。案例 6-1 的 `report()` 就是通过参数与返回值组织四个函数，没有使用任何全局变量。
- 需要在多次调用间累计（如"累计创建多少辆车"），第 11 章会用**类属性**，比 `global` 更清晰。
- 变量名重复（函数内 `d` 与全局 `d`）不会报错，但极易误读，务必避免。

### 6.5.4 随堂示例：局部变量与全局变量互不干扰

**核心要点**：验证函数内部的变量是局部的，函数结束后即消失；函数外定义的同名变量不会被函数内部修改（除非显式声明 `global`，本章不使用）。

In [ ]:
total = 0                     # 全局变量

def calc(n):
    total = n * 2             # 这里是新的局部变量，与全局同名但无关
    inner = "函数内可见"
    return total, inner

value, note = calc(5)
print("函数返回：", value, note)
print("全局 total 仍为：", total)

# 局部变量在函数外不存在，取消下一行注释会报 NameError
# print(inner)

**预期输出**：

```text
函数返回： 10 函数内可见
全局 total 仍为： 0
```

**变式代码（迁移练习）**：相对随堂示例用一次调用演示同名局部，本变式对两条路段连续调用，验证同名局部变量每次都新建、且全局 `total` 自始至终为 0。

In [ ]:
total = 0                                 # 全局 total
def audit(plate, speed):
    total = speed - 120                    # 新建同名局部 total，与全局无关
    flag = "超速" if total > 0 else "正常"
    return plate, flag, total
r1 = audit("京A12345", 132)
r2 = audit("京B67890", 78)
print("两次局部 total：", r1, r2)
print("全局 total 仍是：", total)           # 未被函数内赋值改动

**预期输出**：

```text
两次局部 total： ('京A12345', '超速', 12) ('京B67890', '正常', -42)
全局 total 仍是： 0
```

### 6.5.5 实操步骤（上机跟随）

1. 顶层 `total = 0` 是全局；进入 `audit` 后 `total = speed - 120` 只新建局部绑定，两条路段分别得 12 与 -42
2. 两次调用的局部 `total` 互不影响（`r1` 为 12、`r2` 为 -42），说明栈帧销毁后局部值不残留
3. 最后打印全局 `total` 仍是 0，证明函数内同名赋值没有改动全局变量
4. 若改成先 `print(total)` 再 `total = ...`，预期 `UnboundLocalError`，因为同名已被编译期判为局部
5. 把 `total` 改由参数传入、`flag` 由返回值带出，验证「参数进、结果出」是跨函数通信正解

### 6.5.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| 函数内改了 `total`，外面还是 0 | 函数内 `=` 只新建局部绑定，不改全局引用 | 跨函数传递用参数与返回值，不靠全局变量 |
| `UnboundLocalError` 局部变量赋值前被引用 | 同一函数里既读又赋值同名变量，编译期已判为局部 | 把读取放到赋值之后，或换一个变量名 |
| 以为 `global` 可以随便用 | `global` 让函数产生隐式副作用、难以单独测试 | 本章统一参数进返回值出，累计状态留到第 11 章类属性 |
| 多个函数靠同名全局传数据，结果串味 | 全局只有一份，所有调用共享 | 每个函数显式声明自己需要的参数 |

## 6.6 知识点 6.5　递归

### 6.6.1 核心概念

递归是"函数调用自己"。它适用于**层级深度不固定**的结构，例如路网树（干道 → 支路 → 路口）。

**递归三要素**（缺一不可）：

1. **终止条件**：什么情况下不再递归，直接返回；
2. **递推关系**：本层结果如何由"子问题的结果"组合而成；
3. **参数收敛**：每次递归传入的规模必须变小，否则无限递归。

**原理补充（概念深化）**：每次递归调用都会压入一个全新栈帧，拥有独立的参数与局部变量；「递推」是不断压栈，「回归」是逐层弹栈并把子结果交回上一层，因此递归天生适合深度不固定的树，而处理线性列表只是白白多占栈帧。终止条件必须写在最前：它保证某层不再压栈而直接返回确定值，弹栈才有起点；缺了它或参数不收敛，栈帧会无限增长，直到触到默认约 1000 层上限抛 `RecursionError`。常见误解是「函数里放个 `total = 0` 就能累加全部」——其实每层栈帧的 `total` 都各自新建，只有用 `s += f(child)` 接住子层返回值，结果才沿弹栈方向汇总；这正是「累加变量写在递归层内部会偏小」的根因。写递归前先确认三件事：终止分支、递推式、以及每次传入的规模确实变小。

### 6.6.2 语法要点

```text
def count_nodes(node):
    total = 1
    for child in node["children"]:
        total += count_nodes(child)   # 递推：规模变小
    return total
```

- 递归调用必须出现在 `return` 或赋值中，结果要参与组合；
- 递归深度默认上限约 1000 层，超限抛 `RecursionError`；
- 递归函数的文档字符串应写明"参数含义 + 返回什么"，否则很难读。

### 6.6.3 常见用法与易错点

| 易错点 | 现象 | 修正 |
|---|---|---|
| 忘记终止条件 | `RecursionError: maximum recursion depth exceeded` | 先写终止分支 |
| 参数未变小 | 同上 | 传入 `child` 而非 `node` |
| 累加变量写在递归内 | 每层都从 0 开始，结果偏小 | 初始值在本层设，累加子层返回值 |
| 用递归处理线性数据 | 能跑但没必要 | 列表/字典的遍历用循环 |

---

### 6.6.4 随堂示例：递归统计路网末端节点

**核心要点**：验证递归三要素：终止条件（没有子节点即返回 1）、递推（对每个子节点调用自身）、回归（累加子结果）。

In [ ]:
net = {
    "name": "京港澳干道",
    "children": [
        {"name": "杜家坎", "children": [
            {"name": "A1", "children": []},
            {"name": "A2", "children": []},
        ]},
        {"name": "岳各庄", "children": [
            {"name": "B1", "children": []},
        ]},
    ],
}

def count_leaves(node):
    """递归统计末端路口数"""
    if len(node["children"]) == 0:      # 终止条件
        return 1
    total = 0
    for child in node["children"]:
        total += count_leaves(child)    # 递推
    return total

print("末端路口数：", count_leaves(net))
print("干道直属支路：", len(net["children"]))

**预期输出**：

```text
末端路口数： 3
干道直属支路： 2
```

**变式代码（迁移练习）**：相对随堂示例统计叶子个数，本变式改为累加每个叶子节点携带的车流量数值，验证递归返回值必须参与表达式组合才能沿弹栈方向汇总。

In [ ]:
# 路网树：节点有 children 与 cars，叶子 children 为空
net = {"cars": 0, "children": [{"cars": 0, "children": [{"cars": 820, "children": []}, {"cars": 1100, "children": []}]}, {"cars": 760, "children": []}]}
def total_cars(node):                     # 三要素：终止 / 递推 / 参数收敛
    if len(node["children"]) == 0:
        return node["cars"]
    s = node["cars"]
    for child in node["children"]:
        s += total_cars(child)            # 子层结果沿弹栈方向回传累加
    return s
print("全网车流：", total_cars(net))

**预期输出**：

```text
全网车流： 2680
```

### 6.6.5 实操步骤（上机跟随）

1. `total_cars` 遇到 `children` 为空的叶子直接 `return node["cars"]`，这是终止分支、不再压栈
2. 根节点与支路把本层 `node["cars"]` 作初值，再用 `s += total_cars(child)` 累加各子树结果
3. 全网三个叶子 820、1100、760 汇总输出 `全网车流： 2680`，验证子结果沿弹栈方向回传
4. 把 `s = node["cars"]` 误写进 `for` 内部，每层只剩最后一个子结果，合计会偏小
5. 删掉 `if` 终止分支再运行，预期 `RecursionError`，说明没有终止条件栈帧会无限增长

### 6.6.6 常见问题 FAQ

| 现象 | 根因 | 处理 |
|---|---|---|
| `RecursionError: maximum recursion depth exceeded` | 缺终止条件或参数未收敛，栈帧无限压入 | 先写 `if` 终止分支，递归时传入更小的 `child` |
| 累加结果偏小 | 把累加器初始化写进递归层内部，每层从 0 重来 | 本层设初值，用 `+=` 接住子层返回值 |
| 递归能跑但很慢、像没必要 | 用递归处理线性列表，白白多占栈帧 | 线性数据用 `for` 循环，树形层级才用递归 |
| 子层算出的值最后丢了 | 递归调用结果没 `return`、也没参与表达式组合 | 把 `f(child)` 放进 `return` 或 `+=` 中 |

# 第二部分　代码示范

## 案例 6-1　交通计算函数库

- **覆盖知识点**：6.1（定义与调用）、6.2（默认参数、位置参数）、6.3（多返回值）、6.4（无全局变量）
- **源文件**：`case1_交通计算函数库.py`
- **难度**：基础　**建议用时**：30 分钟

### 任务情境

监测平台需要反复计算通行能力、车流密度、拥堵等级，并生成一条完整的站点简报。要求把这些计算分别封装为函数，主程序只负责调用。

### 任务要求

1. 定义 `capacity(lanes, per_lane=2200)`，`per_lane` 用默认参数；
2. 定义 `density(cars, length_km, lanes)`，返回浮点密度；
3. 定义 `judge(speed, dens)`，返回等级字符串，覆盖四个分支；
4. 定义 `report(...)`，内部调用上面三个函数，**返回多个值**（站点、密度、等级）；
5. 主程序演示默认参数、位置参数与多返回值的解包。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；车道数、车速、车辆数、长度与站点名均为主程序字面量 |
| 类型 | `capacity()` 两参 `int`；`density()` 的 `length_km` 为 `float`，余为 `int`；`road` 为 `str` |
| 取值范围 | 车道 3 条、默认值 2 条；车速 35 与 15；车辆 96；长度 2.5 km；密度 12.8 |
| 非法值示例 | `capacity("3")`、`per_lane` 写在 `lanes` 前、`judge()` 传 `None`、`report()` 只给 4 个实参 |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 恰好 8 行，无空行；`print()` 调用 8 次 |
| 逐行格式 | 第 1 行为通行能力；第 4 行为 `判定(35km/h, 18)： 拥堵`；第 6—8 行为站点、密度、等级 |
| 精度 / 单位 | 通行能力为整数、辆/h；车速整数、km/h；密度保留 2 位小数（12.8） |
| 判定方式 | 与「运行结果」逐字比对 8 行；核对第 3、7 行密度均为 12.8 |

### 参考代码

In [ ]:
# ---------- 第 06 章 函数 · 案例 6-1：交通计算函数库 ----------
# 对应知识点：6.1 定义与调用、6.2 位置参数与默认参数、6.3 返回值与多返回值、6.4 不使用全局变量
# 功能说明：把通行能力、车流密度、拥堵判定封装为函数，由主程序组合调用并解包多返回值

# ---------- 函数定义 ----------
def capacity(lanes, per_lane=2200):
    """理论通行能力（辆/h），per_lane 默认 2200"""
    return lanes * per_lane


def density(cars, length_km, lanes):
    """车流密度（辆/(km·车道)）"""
    return cars / (length_km * lanes)


def judge(speed, dens):
    """按车速与密度判定拥堵等级"""
    if speed >= 60 and dens < 10:
        return "畅通"
    elif speed >= 40:
        return "缓行"
    elif speed >= 20:
        return "拥堵"
    return "严重拥堵"


def report(road, speed, cars, length_km, lanes):
    """生成站点简报，返回三个值"""
    d = density(cars, length_km, lanes)
    level = judge(speed, d)
    return road, round(d, 2), level


# ---------- 主程序 ----------
print("3 车道通行能力：", capacity(3))          # 使用默认 per_lane
print("2 车道(匝道 1800)：", capacity(2, 1800))  # 覆盖默认值
print("密度：", round(density(96, 2.5, 3), 2))
print("判定(35km/h, 18)：", judge(35, 18))
print("判定(15km/h, 45)：", judge(15, 45))

# 多返回值：按位置解包
site, dens, level = report("京港澳 K12", 35, 96, 2.5, 3)
print("站点：", site)
print("密度：", dens)
print("等级：", level)

### 运行结果

```text
3 车道通行能力： 6600
2 车道(匝道 1800)： 3600
密度： 12.8
判定(35km/h, 18)： 拥堵
判定(15km/h, 45)： 严重拥堵
站点： 京港澳 K12
密度： 12.8
等级： 拥堵
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `def capacity(lanes, per_lane=2200)` | 6.1、6.2 | 默认参数放末尾，调用可省略 |
| `capacity(3)` / `capacity(2, 1800)` | 6.2 | 使用默认值 / 位置参数覆盖默认值 |
| `return lanes * per_lane` | 6.3 | 用返回值而非 `print`，结果可参与后续计算 |
| `judge()` 三个 `return` | 6.3 | 提前返回：命中即结束，最后一个 `return` 无需 `else` |
| `report()` 内部三行 | 6.3、6.4 | 通过参数与返回值组织调用，**不使用全局变量** |
| `return road, round(d, 2), level` | 6.3 | 返回元组 |
| `site, dens, level = report(...)` | 6.3 | 解包，变量数与返回值数一致 |

**改一改**：把 `capacity` 的定义改成 `def capacity(per_lane=2200, lanes)`，运行看语法错误；再把 `judge()` 最后一个 `return` 前加上 `else:`，对比两种写法的可读性。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 第 2 行显示 4400 | 失败：第 2 个实参写成 2200，默认值未被覆盖 | 看主程序第 2 条调用 |
| 第 4 行显示「缓行」 | 失败：`speed >= 40` 分支提前命中，等级判错 | 逐条比对各 `return` 前的比较式 |
| 运行报 `TypeError: cannot unpack non-iterable NoneType object` | 失败：`report()` 缺少 `return` | 看 `report()` 最后一行是否为 `return` |
| 运行报 `SyntaxError: non-default argument follows default argument` | 失败：默认参数放到了无默认参数之前 | 看 `capacity` 的形参顺序 |
| 运行报 `ValueError: not enough values to unpack` | 失败：解包变量数与返回值个数不符 | 数等号左侧变量与 `return` 后元素个数 |

## 案例 6-2　递归统计路网节点

- **覆盖知识点**：6.5（递归三要素）、6.3（返回累计值）、5.5（嵌套字典）
- **源文件**：`case2_递归统计路网.py`
- **难度**：综合　**建议用时**：30 分钟

### 任务情境

路网是典型的树形结构：干道下挂支路，支路下挂路口。运营方需要统计任意一级节点的"下属节点总数"与"末端路口数"。由于层级深度不固定，必须用递归。

### 任务要求

1. 用嵌套字典表示路网，每个节点含 `type`、`name`、`children`；
2. 编写 `count_nodes(node)` 统计包含自身在内的全部节点数；
3. 编写 `count_leaves(node)` 统计末端路口数，`children` 为空即为叶子；
4. 演示递归的**终止条件**与**递推关系**。

### 输入规格

| 项 | 要求 |
|---|---|
| 数据来源 | 无外部输入；`road_net` 整体写在源码中，含 1 干道、2 支路、3 路口 |
| 类型 | 节点为 `dict`，字段依次 `str`、`str`、`list`；叶子 `children` 为空列表；`depth` 为 `int` |
| 取值范围 | 节点 6；末端路口 3；深度 3；`max_depth()` 的 `depth` 默认 1 |
| 非法值示例 | 缺 `children` 键、`children` 写成 `None`、`road_net` 改成列表、递归传 `depth - 1` |

### 输出规格

| 项 | 要求 |
|---|---|
| 行数 | 恰好 4 行，无空行 |
| 逐行格式 | 第 1—4 行依次为节点总数 6、末端路口 3、深度 3、直属支路 2 |
| 精度 / 单位 | 全为整数，无小数与单位 |
| 判定方式 | 与「运行结果」逐字比对 4 行；核对第 1 行分项求和与第 4 行元素个数 |

### 参考代码

In [ ]:
# ---------- 第 06 章 函数 · 案例 6-2：递归统计路网节点 ----------
# 对应知识点：6.5 递归三要素（终止条件 / 递推 / 回归）、6.3 返回累计值、5.5 嵌套字典遍历
# 功能说明：递归遍历「干道—支路—路口」树结构，统计节点总数、末端路口数与层级深度

# 路网树：干道 -> 支路 -> 路口
road_net = {
    "type": "干道", "name": "京港澳高速",
    "children": [
        {"type": "支路", "name": "杜家坎", "children": [
            {"type": "路口", "name": "A1", "children": []},
            {"type": "路口", "name": "A2", "children": []},
        ]},
        {"type": "支路", "name": "岳各庄", "children": [
            {"type": "路口", "name": "B1", "children": []},
        ]},
    ],
}


def count_nodes(node):
    """统计本节点及其所有下级节点总数"""
    total = 1                       # 自己算 1 个
    for child in node["children"]:
        total += count_nodes(child)  # 递归处理每个子节点
    return total


def count_leaves(node):
    """统计末端路口数：没有子节点即为路口"""
    if len(node["children"]) == 0:   # 终止条件
        return 1
    total = 0
    for child in node["children"]:
        total += count_leaves(child)  # 递推
    return total


def max_depth(node, depth=1):
    """统计路网层级深度"""
    if len(node["children"]) == 0:
        return depth
    best = depth
    for child in node["children"]:
        d = max_depth(child, depth + 1)
        if d > best:
            best = d
    return best


print("路网节点总数：", count_nodes(road_net))
print("末端路口总数：", count_leaves(road_net))
print("层级深度：", max_depth(road_net))
print("干道直属支路数：", len(road_net["children"]))

### 运行结果

```text
路网节点总数： 6
末端路口总数： 3
层级深度： 3
干道直属支路数： 2
```

### 代码解读

| 代码段 | 对应知识点 | 作用 |
|---|---|---|
| `road_net` 嵌套字典 | 5.5 | 树的三个字段：类型、名称、子节点列表 |
| `total = 1` | 6.5 | 递归初始值在本层设，代表"自己" |
| `total += count_nodes(child)` | 6.5 | 递推：把每个子树的节点数累加进来 |
| `if len(node["children"]) == 0: return 1` | 6.5 | `count_leaves` 的**终止条件** |
| `max_depth(node, depth=1)` | 6.2、6.5 | 用默认参数携带初始层数，`depth + 1` 实现"携带状态" |
| `best` 与 `d > best` | 6.5 | 多层子树中取最大深度，等价于第 3 章的极值维护 |

**改一改**：把 `count_nodes` 里的 `total = 1` 改成 `total = 0`，观察结果是否变成 5（漏算根节点）；再注释掉 `count_leaves` 的终止条件，运行后记录 `RecursionError` 的提示信息。

### 失败判定

| 现象 | 判定结论 | 定位方法 |
|---|---|---|
| 第 1 行显示 5 | 失败：`count_nodes` 的 `total = 1` 被改成 0，漏算根节点 | 看 `count_nodes` 首行的初值 |
| 运行报 `RecursionError: maximum recursion depth exceeded` | 失败：终止条件被删或 `children` 判断写错 | 看 `count_leaves` 的第一个 `if` |
| 运行报 `KeyError: 'children'` | 失败：某个节点缺少 `children` 键 | 核对节点的三个字段 |
| 第 3 行显示 1 | 失败：递归时未把 `depth + 1` 传下去 | 看递归调用处的第二个实参 |
| 第 4 行显示 3 | 失败：把后代支路数当成直属支路数 | 数 `road_net["children"]` 元素个数 |

---

# 第三部分　布置练习

## 练习 6-1　用可变参数与关键字参数实现计费

- **对应知识点**：6.2（`*args`、默认参数、关键字调用）、6.3（返回值）
- **提交文件**：`practice1_可变参数计费.py`
- **难度**：综合　**建议用时**：25 分钟

### 任务情境

你被派去给收费系统的结算模块补三个函数：一个汇总任意多个断面送来的车流量、空参数时返回 0，一个给出保留 1 位小数的平均值，还有一个按「里程 × 费率 × 折扣」算通行费，费率默认 0.5、折扣默认 1.0。结算员常只改折扣，所以必须支持关键字调用，三个函数都要有文档字符串。

### 题目

收费系统需要：① 汇总任意多个路段的车流量；② 按"里程 × 费率 × 折扣"计算通行费，费率与折扣可省略。

### 要求

1. 定义 `total_flow(*flows)`，支持任意个数参数，无参数时返回 0；
2. 定义 `avg_flow(*flows)`，返回平均值（保留 1 位小数），无参数时返回 0；
3. 定义 `fee(km, rate=0.5, discount=1.0)`，返回保留 2 位小数的费用；
4. 至少演示一次关键字参数调用（如 `fee(km=120, discount=0.95)`）。

### 提示

- `*flows` 在函数内部是一个元组，可直接用 `len()` 与 `sum()`；无参数时 `len(flows) == 0`，需单独返回，避免除零。
- 关键字调用可以让"只改折扣、不动费率"的意图更清楚：`fee(km=120, discount=0.95)`。
- 三个函数的顺序建议按"通用 → 专用"排列，主程序集中在文件末尾。

### 自检要点（不含答案）

- [ ] 无参数调用 `total_flow()` 与 `avg_flow()` 都返回 0，不报错；
- [ ] `fee(120)` 得 60.0，`fee(120, 0.6)` 得 72.0；
- [ ] 关键字调用结果与位置调用一致；
- [ ] 每个函数都有文档字符串。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 不改源码，直接运行答案 6-1 | 7 行输出，总量 2680、均值 842.5、费用 60.0、72.0、57.0 | 与答案 6-1「运行结果」逐字比对 7 行 |
| T2 | 边界 | `avg_flow(820, 1100, 760, 690)` 改为 `avg_flow()` | 第 3 行由 `842.5` 变为 `0`，不抛 `ZeroDivisionError` | 核对第 3 行数值，进程正常结束 |
| T3 | 异常 | 把 `fee` 的参数改为 `(rate=0.5, km, discount=1.0)` | 启动即报 `SyntaxError: non-default argument follows default argument` | 看报错信息含 `non-default argument` |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| `*args` 使用 | 3 个与 0 个实参各调一次 | 任意个数可合计，无参返回 0 | 3 |
| 默认参数 | 读函数定义 | 无默认值参数在前，可覆盖 | 2 |
| 关键字调用 | 读调用行 | 至少一次，结果与位置一致 | 2 |
| 文档字符串 | 数三引号 | 三个函数都有 | 1 |
| 数值正确 | 读第 1、3、4、6、7 行 | 2680、842.5、60.0、57.0、81.0 | 2 |
| 合计 | — | — | 10 |

## 练习 6-2　函数拆分重构拥堵播报

- **对应知识点**：6.1（定义）、6.3（返回值）、6.4（无全局变量）、第 3 章分支
- **提交文件**：`practice2_播报重构.py`
- **难度**：综合　**建议用时**：30 分钟

### 任务情境

你是路网运行监测岗的新人，接手了上一班的拥堵播报脚本：5 条路段的车速与播报文字全挤在一段代码里，判定、建议、输出混在一起。班长要求按「判定、建议、组装」拆成三个函数，都不写 `print`，主程序只调用组装函数批量输出 5 行；车速 66 的京港澳 K12 必须给出「畅通」和「保持当前配时」。

### 题目

把第 3 章"拥堵分级"的整段程序重构为三个函数：`judge_level(speed)`、`advice(level)`、`build_report(road, speed)`，主程序只调用 `build_report()`。

### 要求

1. `judge_level()` 只负责判定，返回等级字符串；
2. `advice()` 只负责映射建议，用 `if-elif` 覆盖四个等级；
3. `build_report()` 调用前两个函数，返回"路段 + 等级 + 建议"组成的完整字符串；
4. 主程序对 5 条路段批量调用并输出。

### 提示

- 重构的判断标准是"每个函数能否用一句话说清职责"。若某个函数既判定又输出，说明职责不单一。
- 三个函数都不应写 `print`：输出统一放主程序，函数只 `return`。
- 批量调用需要两个等长列表配合下标遍历（本章尚未学 `zip`，用 `range(len(...))`）。

### 自检要点（不含答案）

- [ ] 三个函数均无 `print`，只有 `return`；
- [ ] `build_report("京港澳 K12", 66)` 返回包含"畅通"与"保持当前配时"的字符串；
- [ ] 主程序输出 5 行，无重复代码；
- [ ] 函数之间无全局变量通信。

### 测试用例表

| 编号 | 类别 | 输入 | 关键预期输出 | 判定方式 |
|---|---|---|---|---|
| T1 | 正常 | 不改源码，直接运行答案 6-2 | 5 行播报，第 1 行三段为 `京港澳 K12`、`畅通`、`保持当前配时` | 与答案 6-2「运行结果」逐字比对 5 行 |
| T2 | 边界 | 把 `speeds` 第 3 个元素 28 改为 40 | 第 3 行变为机场高速 K3 判缓行、建议绿灯延长 5 秒 | 核对第 3 行的等级与建议 |
| T3 | 异常 | 删除 `judge_level` 最后一个 `return "严重拥堵"` | 处理到第 4 条记录时抛 `TypeError`，前 3 行已输出 | 看报错信息中含 `TypeError` 与 `NoneType` |

### 验收标准表

| 检查项 | 判定方法 | 通过标准 | 分值 |
|---|---|---|---|
| 三个函数职责单一 | 读函数体 | 均无 `print`，只有 `return`，判定与建议分离 | 4 |
| 分支完整 | 传 66、45、28、15 | 得到畅通、缓行、拥堵、严重拥堵 | 3 |
| 组装结果 | 读第 1 行 | 三段齐全，竖线分隔 | 2 |
| 批量输出 | 数输出行数 | 恰好 5 行 | 1 |
| 合计 | — | — | 10 |

---

# 第四部分　习题答案

## 答案 6-1　用可变参数与关键字参数实现计费

> 对应 **练习 6-1**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 06 章 函数 · 练习 6-1 参考答案：可变参数与关键字参数计费 ----------
# 对应知识点：6.2 *args 可变参数、默认参数、关键字调用、6.3 返回值
# 功能说明：用可变参数合计任意多个路段的车流，用关键字参数控制费率与折扣计算通行费

def total_flow(*flows):
    """任意多个路段的车流总量"""
    if len(flows) == 0:
        return 0
    return sum(flows)


def avg_flow(*flows):
    """任意多个路段的平均流量"""
    if len(flows) == 0:
        return 0
    return round(sum(flows) / len(flows), 1)


def fee(km, rate=0.5, discount=1.0):
    """通行费 = 里程 × 费率 × 折扣"""
    return round(km * rate * discount, 2)


print("三路段总量：", total_flow(820, 1100, 760))
print("无参数总量：", total_flow())
print("四路段均值：", avg_flow(820, 1100, 760, 690))
print("默认费率：", fee(120))
print("指定费率：", fee(120, 0.6))
print("关键字调用：", fee(km=120, discount=0.95))
print("全部关键字：", fee(km=200, rate=0.45, discount=0.9))

### 运行结果

```text
三路段总量： 2680
无参数总量： 0
四路段均值： 842.5
默认费率： 60.0
指定费率： 72.0
关键字调用： 57.0
全部关键字： 81.0
```

### 解析

1. **`*flows` 的类型**：函数内部 `flows` 是元组，`sum()` 与 `len()` 都可直接用；不能 `append`（元组不可变）。
2. **零参数保护**：`avg_flow()` 若不判空，`sum(())/0` 会抛 `ZeroDivisionError`。先判 `len(flows) == 0` 是标准写法。
3. **关键字调用的价值**：`fee(km=120, discount=0.95)` 明确表达了"里程 120、只改折扣"，读者无需回忆参数顺序，这是 6.2 强调的可读性优势。
4. **默认参数顺序**：`fee(km, rate=0.5, discount=1.0)` 中 `km` 无默认值必须在前，两个有默认值的在后——顺序不能颠倒。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| `ZeroDivisionError` | `avg_flow()` 未判空 | 先判 `len(flows) == 0` |
| `SyntaxError` | `def fee(rate=0.5, km)` | 无默认值的参数放前面 |
| 结果为 60.0 而非 57.0 | 未传 `discount` | 关键字调用传全 |
| `AttributeError: 'tuple' object has no attribute 'append'` | 把 `flows` 当列表 | 元组不可变，改用 `sum()` |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| `*args` 使用 | 3 | 能接收任意个数、无参不报错 |
| 默认参数 | 2 | 顺序正确、可覆盖 |
| 关键字调用 | 2 | 至少一次，结果正确 |
| 文档字符串 | 1 | 三个函数都有 |
| 数值正确 | 2 | 2680 / 842.5 / 60.0 / 57.0 / 81.0 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 7 行输出与期望逐字一致 | 正常路径，直接复现本答案 |
| T2 | 无参调用返回 0 而不报错 | 见解析第 2 条，先判长度再求平均 |
| T3 | 参数顺序颠倒时启动即报语法错误 | 见解析第 4 条参数顺序规则 |

## 答案 6-2　函数拆分重构拥堵播报

> 对应 **练习 6-2**（第三部分）

### 参考代码

In [ ]:
# ---------- 第 06 章 函数 · 练习 6-2 参考答案：函数拆分重构拥堵播报 ----------
# 对应知识点：6.1 定义与调用、6.3 返回值、6.4 不依赖全局变量、3.1 分支
# 功能说明：把「判定等级—给出建议—拼装播报」拆成三个函数，循环批量生成 5 条路段播报

def judge_level(speed):
    if speed >= 60:
        return "畅通"
    elif speed >= 40:
        return "缓行"
    elif speed >= 20:
        return "拥堵"
    return "严重拥堵"


def advice(level):
    if level == "畅通":
        return "保持当前配时"
    elif level == "缓行":
        return "绿灯延长 5 秒"
    elif level == "拥堵":
        return "启动绿波协调，发布绕行提示"
    return "发布一级诱导，通知警力现场疏导"


def build_report(road, speed):
    level = judge_level(speed)
    return road + " | " + level + " | " + advice(level)


roads = ["京港澳 K12", "京开 K8", "机场高速 K3", "京承 K20", "五环西段"]
speeds = [66, 45, 28, 15, 52]

for i in range(len(roads)):
    print(build_report(roads[i], speeds[i]))

### 运行结果

```text
京港澳 K12 | 畅通 | 保持当前配时
京开 K8 | 缓行 | 绿灯延长 5 秒
机场高速 K3 | 拥堵 | 启动绿波协调，发布绕行提示
京承 K20 | 严重拥堵 | 发布一级诱导，通知警力现场疏导
五环西段 | 缓行 | 绿灯延长 5 秒
```

### 解析

1. **职责三步切分**：`judge_level` 只做"指标 → 等级"，`advice` 只做"等级 → 建议"，`build_report` 只做"组装文本"。每个函数都能用一句话说清，符合 6.1 的判断标准。
2. **为什么要拆**：若等级标准调整（如 60 改为 65），只改 `judge_level` 一处；若话术调整，只改 `advice`。原程序把判定、建议、输出混在一起，改一处要读全段。
3. **函数不打印**：三个函数都只 `return`，输出集中到主程序。这让函数可以在第 8 章被写入文件、在第 13 章被图形界面调用，而不必修改函数本身。
4. **批量遍历**：本章未学 `zip`，用 `range(len(roads))` 配合下标取两个列表的值，是第 5 章"下标访问"的延续。

### 常见错误

| 错误现象 | 原因 | 修正 |
|---|---|---|
| 函数里有 `print` | 职责不单一 | 改为 `return` |
| 返回 `None` | 忘记 `return` | 每个分支都要 `return` |
| 建议与等级错位 | `advice` 的判断顺序颠倒 | 按等级字符串精确匹配 |
| 只输出 1 行 | 主程序未循环 | 用 `for` 遍历 |

### 评分要点（供批改）

| 项 | 分值 | 观察点 |
|---|---|---|
| 三个函数职责清晰 | 4 | 无 `print`、无重复判定 |
| 分支完整 | 3 | 四个等级全覆盖 |
| 组装结果 | 2 | 路段、等级、建议三段齐全 |
| 批量输出 | 1 | 5 行，速度快慢对应正确 |

### 与测试用例的对应

| 用例 | 本答案的对应结果 | 说明 |
|---|---|---|
| T1 | 5 行播报与期望逐字一致 | 正常路径，直接复现本答案 |
| T2 | 车速 40 判为缓行 | `speed >= 40` 兜住边界值 |
| T3 | 缺最后一条 `return` 时抛 `TypeError` | 分支不全返回 `None`，拼接即报错 |

---

## 本章小结

- **定义与调用**：`def` 只登记，调用才执行；函数名 + 圆括号；先定义后调用。
- **四类参数**：位置、关键字、默认（**放末尾**）、`*args`（函数内为元组）。
- **返回值**：`return` 立即结束函数；多返回值本质是元组，可解包；函数应"返回结果"而非"打印结果"。
- **作用域**：函数内赋值即局部变量；跨函数通信靠参数与返回值，不靠全局变量。
- **递归**：终止条件 + 递推关系 + 参数收敛，三者缺一不可。

## 自检清单

- [ ] 能写出带默认参数、`*args`、多返回值的函数；
- [ ] 知道默认参数为何必须放末尾；
- [ ] 能说出 `print` 与 `return` 的区别，并判断何时必须用 `return`；
- [ ] 能识别"函数内赋值即新建局部变量"的现象；
- [ ] 能写出带终止条件的递归函数，并解释每层在算什么；
- [ ] 能把一段长程序按"单一职责"拆成三个函数。

## 前后章节衔接

- 本章的函数是**第 7 章高阶函数**的操作对象：`map` / `filter` / `sorted` 都要"把函数当参数传"。
- 本章的文档字符串在**第 10 章**成为模块说明的来源（`模块.__doc__`）。
- 本章的递归将在**第 12 章**以"继承链上的 `super()` 调用"形式再次出现，思路仍是"分层处理、逐层返回"。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：Trae调试功能](../05A_Trae调试功能/05A_Trae调试功能.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[下一章：高级函数 ▶](../07高级函数/07_高级函数.ipynb)